In [2]:
# ============================================================
# 03_pyspark_pipeline.ipynb
# PySpark ML Pipeline - NYC Taxi Dataset
# ============================================================

# 1. Imports

import time
import requests
import pandas as pd

from pydantic import ByteSize
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator


In [3]:
# 2. Start Spark session

spark = (
    SparkSession.builder
    .appName("BDCC PySpark Taxi Pipeline")
    .getOrCreate()
)

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/29 15:18:40 WARN Utils: Your hostname, MacBook-Air-4.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.231 instead (on interface en0)
26/05/29 15:18:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 15:18:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# 3. Configuration

RUN_MODE = "local"

if RUN_MODE == "local":
    DATA_PATH = "../data/raw/taxi/yellow_tripdata_2026-01.parquet"
else:
    DATA_PATH = "gs://your-bucket/taxi/yellow_tripdata_2026-01.parquet"

LOOKUP_PATH = "../data/raw/taxi/taxi_zone_lookup.csv"

SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

TARGET = "fare_amount"

FEATURES = [
    "trip_distance",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "payment_type",
]

FEATURES_COL = "features"
LABEL_COL = "label"


In [6]:
from collections.abc import Generator
from contextlib import contextmanager
from datetime import timedelta
from pathlib import Path
from time import perf_counter
from typing import TypedDict

class Duration(TypedDict):
    duration: timedelta


def _mem_str(path: str) -> str:
    path = Path(path)
    size = path.stat().st_size
    return ByteSize(size).human_readable(decimal=True)


def download_if_missing(path: str, url: str) -> None:
    path = Path(path)
    if path.exists():
        return

    print(f"Downloading {path.name}...")
    response = requests.get(url)
    response.raise_for_status()
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded {path}, {_mem_str(path)}")


@contextmanager
def timeit(name: str) -> Generator[dict]:
    result = {"duration": timedelta()}
    start_time = perf_counter()
    yield result
    end_time = perf_counter()
    duration = timedelta(seconds=end_time - start_time)
    print(f"{name} took {duration}")
    result["duration"] = duration


In [7]:
# 4. Load data

download_if_missing(DATA_PATH, SOURCE_URL)
download_if_missing(LOOKUP_PATH, LOOKUP_URL)

with timeit("Loading data"):
    df = spark.read.parquet(DATA_PATH)

location_lookup = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(LOOKUP_PATH)
)

df.show(5)


Loading data took 0:00:00.743765
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|   

In [8]:
# 5. Basic dataset inspection

print("Columns:")
print(df.columns)

print("\nSchema:")
df.printSchema()

print("\nNumber of partitions:")
print(df.rdd.getNumPartitions())


Columns:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = t

In [9]:
# 6. Preprocessing

df = df.select(*(FEATURES + [TARGET]))

df = df.dropna()

df = df.filter(
    (F.col("fare_amount") > 0) &
    (F.col("trip_distance") > 0) &
    (F.col("passenger_count") > 0)
)

df.show(5)


+-------------+---------------+------------+------------+------------+-----------+
|trip_distance|passenger_count|PULocationID|DOLocationID|payment_type|fare_amount|
+-------------+---------------+------------+------------+------------+-----------+
|         0.97|              1|         239|         238|           1|        7.2|
|         5.58|              4|         142|         209|           1|       38.7|
|         2.33|              2|         144|         137|           1|       14.2|
|          1.3|              1|         142|          50|           2|       11.4|
|         5.34|              1|         161|          45|           1|       37.3|
+-------------+---------------+------------+------------+------------+-----------+
only showing top 5 rows


In [10]:
def benchmark_pyspark(data_path: str, lookup_path: str):
    metrics = {}

    # 1. Read Data
    with timeit("1. Read Data") as duration:
        df = spark.read.parquet(data_path)
        df_lookup = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(lookup_path)
        )
        _ = df.count()
    metrics["1. Read Data"] = duration["duration"]

    # 2. Count Operation
    with timeit("2. Count Operation") as duration:
        _ = df.count()
    metrics["2. Count Operation"] = duration["duration"]

    # 3. Complex Arithmetic Formula
    with timeit("3. Complex Arithmetic") as duration:
        arithmetic_df = df.select(
            (((F.col("fare_amount") + F.col("tip_amount")) * 1.5) / (F.col("passenger_count") + 1)).alias("result")
        )
        _ = arithmetic_df.count()
    metrics["3. Complex Arithmetic"] = duration["duration"]

    # 4. Statistical Standard Deviation
    with timeit("4. Standard Deviation") as duration:
        _ = df.select(F.stddev("trip_distance")).collect()
    metrics["4. Standard Deviation"] = duration["duration"]

    # 5. GroupBy Aggregation
    with timeit("5. GroupBy Mean") as duration:
        _ = (
            df.groupBy("passenger_count")
            .agg(F.mean("fare_amount").alias("mean_fare_amount"))
            .collect()
        )
    metrics["5. GroupBy Mean"] = duration["duration"]

    # 6. Merge/Join with Count
    with timeit("6. Join & Count") as duration:
        df_lookup = df_lookup.withColumn("LocationID", F.col("LocationID").cast(df.schema["PULocationID"].dataType))
        joined = df.join(df_lookup, df["PULocationID"] == df_lookup["LocationID"], how="inner")
        _ = joined.count()
    metrics["6. Join & Count"] = duration["duration"]

    return metrics


In [11]:
benchmarks = benchmark_pyspark(DATA_PATH, LOOKUP_PATH)

benchmarks


1. Read Data took 0:00:00.350333
2. Count Operation took 0:00:00.050068
3. Complex Arithmetic took 0:00:00.066850
4. Standard Deviation took 0:00:00.187848
5. GroupBy Mean took 0:00:00.320805
6. Join & Count took 0:00:00.250183


{'1. Read Data': datetime.timedelta(microseconds=350333),
 '2. Count Operation': datetime.timedelta(microseconds=50068),
 '3. Complex Arithmetic': datetime.timedelta(microseconds=66850),
 '4. Standard Deviation': datetime.timedelta(microseconds=187848),
 '5. GroupBy Mean': datetime.timedelta(microseconds=320805),
 '6. Join & Count': datetime.timedelta(microseconds=250183)}

In [12]:
# 7. Feature vector preparation

assembler = VectorAssembler(
    inputCols=FEATURES,
    outputCol=FEATURES_COL,
    handleInvalid="skip"
)

ml_df = assembler.transform(df).select(
    F.col(FEATURES_COL),
    F.col(TARGET).alias(LABEL_COL)
)

train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

train_df.cache()
test_df.cache()

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


Train rows: 2041282


Test rows: 510569


In [13]:
# 8. Regression pipeline - GBTRegressor
# Note: PySpark MLlib does not use XGBRegressor by default.
# GBTRegressor is the closest native Spark tree boosting alternative.

reg_model = GBTRegressor(
    featuresCol=FEATURES_COL,
    labelCol=LABEL_COL,
    maxIter=50,
    maxDepth=6,
    seed=42
)

with timeit("Regression training") as duration:
    fitted_reg_model = reg_model.fit(train_df)

regression_train_time = duration["duration"]

reg_predictions = fitted_reg_model.transform(test_df)
reg_predictions.select("prediction", LABEL_COL).show(5)


Regression training took 0:00:51.094189
+------------------+-----+
|        prediction|label|
+------------------+-----+
| 34.83243563826971| 25.0|
|14.320014374143884|  3.0|
| 34.83243563826971|  3.0|
| 34.83243563826971| 30.0|
|  9.78436068754016|  3.0|
+------------------+-----+
only showing top 5 rows


26/05/29 15:20:18 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [14]:
# 9. Regression metrics

mae_evaluator = RegressionEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="mae"
)

mse_evaluator = RegressionEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="mse"
)

rmse_evaluator = RegressionEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="rmse"
)

r2_evaluator = RegressionEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="r2"
)

regression_results = {
    "library": "PySpark",
    "task": "regression",
    "model": "GBTRegressor",
    "train_time": regression_train_time,
    "mae": mae_evaluator.evaluate(reg_predictions),
    "mse": mse_evaluator.evaluate(reg_predictions),
    "rmse": rmse_evaluator.evaluate(reg_predictions),
    "r2": r2_evaluator.evaluate(reg_predictions),
}

regression_results


{'library': 'PySpark',
 'task': 'regression',
 'model': 'GBTRegressor',
 'train_time': datetime.timedelta(seconds=51, microseconds=94189),
 'mae': 2.9690991505275637,
 'mse': 44.11983673490703,
 'rmse': 6.642276472332888,
 'r2': 0.8682646693231346}

In [15]:
# 10. Classification pipeline - LogisticRegression

classified_df = df.withColumn(
    "fare_class",
    F.when(F.col(TARGET) < 15, F.lit(0))
     .when(F.col(TARGET) < 40, F.lit(1))
     .otherwise(F.lit(2))
     .cast(IntegerType())
)

classification_ml_df = assembler.transform(classified_df).select(
    F.col(FEATURES_COL),
    F.col("fare_class").alias(LABEL_COL)
)

train_cls_df, test_cls_df = classification_ml_df.randomSplit([0.8, 0.2], seed=42)

train_cls_df.cache()
test_cls_df.cache()

cls_model = LogisticRegression(
    featuresCol=FEATURES_COL,
    labelCol=LABEL_COL,
    maxIter=100
)

with timeit("Classification training") as duration:
    fitted_cls_model = cls_model.fit(train_cls_df)

classification_train_time = duration["duration"]

cls_predictions = fitted_cls_model.transform(test_cls_df)
cls_predictions.select("prediction", LABEL_COL).show(5)


26/05/29 15:20:40 ERROR StrongWolfeLineSearch: Encountered bad values in function evaluation. Decreasing step size to 0.5
26/05/29 15:20:40 ERROR StrongWolfeLineSearch: Encountered bad values in function evaluation. Decreasing step size to 0.5
26/05/29 15:20:40 ERROR StrongWolfeLineSearch: Encountered bad values in function evaluation. Decreasing step size to 0.25
26/05/29 15:20:40 ERROR StrongWolfeLineSearch: Encountered bad values in function evaluation. Decreasing step size to 0.125
26/05/29 15:20:41 ERROR StrongWolfeLineSearch: Encountered bad values in function evaluation. Decreasing step size to 0.09375
26/05/29 15:20:43 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/05/29 15:20:44 ERROR StrongWolfeLineSearch: Encountered bad values in function evaluation. Decreasing step size to 0.5
26/05/29 15:20:44 ERROR StrongWolfeLineSearch: Encountered bad values in function evaluation. Decreasing step size to 0.25
26/05/29 15:20:44 

Classification training took 0:00:10.835864


+----------+-----+
|prediction|label|
+----------+-----+
|       1.0|    1|
|       2.0|    0|
|       1.0|    0|
|       1.0|    1|
|       0.0|    0|
+----------+-----+
only showing top 5 rows


In [16]:
# 11. Classification metrics

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="accuracy"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="f1"
)

classification_results = {
    "library": "PySpark",
    "task": "classification",
    "model": "LogisticRegression",
    "train_time": classification_train_time,
    "accuracy": accuracy_evaluator.evaluate(cls_predictions),
    "precision_weighted": precision_evaluator.evaluate(cls_predictions),
    "recall_weighted": recall_evaluator.evaluate(cls_predictions),
    "f1_weighted": f1_evaluator.evaluate(cls_predictions),
}

classification_results


{'library': 'PySpark',
 'task': 'classification',
 'model': 'LogisticRegression',
 'train_time': datetime.timedelta(seconds=10, microseconds=835864),
 'accuracy': 0.5913872561788907,
 'precision_weighted': 0.5260776364433364,
 'recall_weighted': 0.5913872561788905,
 'f1_weighted': 0.44737175471205076}

In [17]:
# 12. Save results

from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

benchmark_results_df = pd.DataFrame([
    {"operation": operation, "library": "PySpark", "duration": duration}
    for operation, duration in benchmarks.items()
])

ml_results_df = pd.DataFrame([
    regression_results,
    classification_results
])

benchmark_results_df.to_csv("../results/pyspark_benchmark_results.csv", index=False)
ml_results_df.to_csv("../results/pyspark_pipeline_results.csv", index=False)

benchmark_results_df, ml_results_df


(               operation  library               duration
 0           1. Read Data  PySpark 0 days 00:00:00.350333
 1     2. Count Operation  PySpark 0 days 00:00:00.050068
 2  3. Complex Arithmetic  PySpark 0 days 00:00:00.066850
 3  4. Standard Deviation  PySpark 0 days 00:00:00.187848
 4        5. GroupBy Mean  PySpark 0 days 00:00:00.320805
 5        6. Join & Count  PySpark 0 days 00:00:00.250183,
    library            task               model             train_time  \
 0  PySpark      regression        GBTRegressor 0 days 00:00:51.094189   
 1  PySpark  classification  LogisticRegression 0 days 00:00:10.835864   
 
         mae        mse      rmse        r2  accuracy  precision_weighted  \
 0  2.969099  44.119837  6.642276  0.868265       NaN                 NaN   
 1       NaN        NaN       NaN       NaN  0.591387            0.526078   
 
    recall_weighted  f1_weighted  
 0              NaN          NaN  
 1         0.591387     0.447372  )